In [ ]:
import osmium
from rtree import index
from shapely.geometry import Point
import time

# --- Settings ---
PBF_FILE = "Data_collection/nevada-latest.osm.pbf"
SEARCH_RADIUS_M = 300  # meters
DEGREE_BUFFER = SEARCH_RADIUS_M / 111320  # rough meter-to-degree conversion

# --- Index Builder ---
class OSMIndexBuilder(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.idx = {
            'railway': index.Index(),
            'water': index.Index(),
            'footpath': index.Index(),
            'bus_stop': index.Index(),
            'ramp': index.Index(),
            'bike_lane': index.Index()
        }
        self.meta = {k: {} for k in self.idx}
        self.id_counters = {k: 0 for k in self.idx}

    def add(self, key, lat, lon, tags):
        pid = self.id_counters[key]
        pt = Point(lon, lat)
        self.idx[key].insert(pid, pt.bounds)
        self.meta[key][pid] = {'point': pt, 'tags': dict(tags)}
        self.id_counters[key] += 1

    def node(self, n):
        if not n.location.valid():
            return
        lat, lon = n.location.lat, n.location.lon
        tags = n.tags

        if tags.get('railway') in {'rail', 'level_crossing'}:
            self.add('railway', lat, lon, tags)
        elif tags.get('natural') == 'water' or 'waterway' in tags:
            self.add('water', lat, lon, tags)
        elif tags.get('highway') in {'footway', 'path'} or 'sidewalk' in tags:
            self.add('footpath', lat, lon, tags)
        elif tags.get('highway') == 'bus_stop' or tags.get('public_transport') == 'platform':
            self.add('bus_stop', lat, lon, tags)
        elif tags.get('highway') == 'motorway_link' or 'motorway_junction' in tags:
            self.add('ramp', lat, lon, tags)
        elif 'cycleway' in tags or tags.get('bicycle') == 'designated':
            self.add('bike_lane', lat, lon, tags)

# --- Point Evaluator ---
def evaluate_stop(lat, lon, builder):
    pt = Point(lon, lat)
    buf = pt.buffer(DEGREE_BUFFER)

    def nearby(key):
        return [
            builder.meta[key][i]
            for i in builder.idx[key].intersection(buf.bounds)
            if builder.meta[key][i]['point'].within(buf)
        ]

    return {
        'railroad_nearby': len(nearby('railway')) > 0,
        'water_nearby': len(nearby('water')) > 0,
        'sidewalk_or_path': len(nearby('footpath')) > 0,
        'bus_stop_tagged': len(nearby('bus_stop')) > 0,
        'freeway_ramp_nearby': len(nearby('ramp')) > 0,
        'bike_lane_present': len(nearby('bike_lane')) > 0
    }

# --- Run Everything ---
if __name__ == "__main__":
    print("Parsing and indexing OSM data...")
    t0 = time.time()
    builder = OSMIndexBuilder()
    builder.apply_file(PBF_FILE, locations=True)
    print(f"Indexing complete in {time.time() - t0:.1f} seconds.")

    # --- Example coordinate ---
    lat, lon = 36.1523113, -115.1571111
    result = evaluate_stop(lat, lon, builder)

    print(f"\nEvaluation for point ({lat}, {lon}):")
    for k, v in result.items():
        print(f"  {k}: {'YES' if v else 'NO'}")


Parsing and indexing OSM data...
Indexing complete in 246.9 seconds.

Evaluation for point (36.1523113, -115.1571111):
  railroad_nearby: NO
  water_nearby: NO
  sidewalk_or_path: NO
  bus_stop_tagged: NO
  freeway_ramp_nearby: NO
  bike_lane_present: NO


In [ ]:
import osmium
import geojson
import os

OUTPUT_DIR = "extracted_features"
PBF_FILE = "Data_collection/nevada-latest.osm.pbf"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

class GeometryCollector(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.features = {
            'railways': [],
            'water': [],
            'footpaths': [],
            'ramps': [],
            'bike_lanes': []
        }

    def node(self, n):
        if not n.location.valid():
            return
        tags = dict(n.tags)
        lat, lon = n.location.lat, n.location.lon
        geom = geojson.Point((lon, lat))

        if tags.get('railway') in {'rail', 'level_crossing'}:
            self.features['railways'].append((geom, tags))
        elif tags.get('natural') == 'water' or 'waterway' in tags:
            self.features['water'].append((geom, tags))
        elif tags.get('highway') in {'footway', 'path'} or 'sidewalk' in tags:
            self.features['footpaths'].append((geom, tags))
        elif tags.get('highway') == 'motorway_link' or 'motorway_junction' in tags:
            self.features['ramps'].append((geom, tags))
        elif 'cycleway' in tags or tags.get('bicycle') == 'designated':
            self.features['bike_lanes'].append((geom, tags))

    def way(self, w):
        if not w.nodes:
            return
        tags = dict(w.tags)
        coords = []
        for n in w.nodes:
            if n.location.valid():
                coords.append((n.lon, n.lat))

        if len(coords) < 2:
            return

        is_polygon = coords[0] == coords[-1] and len(coords) >= 4
        if is_polygon:
            geom = geojson.Polygon([coords])
        else:
            geom = geojson.LineString(coords)

        if tags.get('railway') in {'rail', 'level_crossing'}:
            self.features['railways'].append((geom, tags))
        elif tags.get('natural') == 'water' or 'waterway' in tags:
            self.features['water'].append((geom, tags))
        elif tags.get('highway') in {'footway', 'path'} or 'sidewalk' in tags:
            self.features['footpaths'].append((geom, tags))
        elif tags.get('highway') == 'motorway_link' or 'motorway_junction' in tags:
            self.features['ramps'].append((geom, tags))
        elif 'cycleway' in tags or tags.get('bicycle') == 'designated':
            self.features['bike_lanes'].append((geom, tags))

def save_geojson(name, features):
    fc = geojson.FeatureCollection([
        geojson.Feature(geometry=geom, properties=tags)
        for geom, tags in features
    ])
    with open(os.path.join(OUTPUT_DIR, f"{name}.geojson"), "w", encoding="utf-8") as f:
        geojson.dump(fc, f, indent=2)

if __name__ == "__main__":
    print(f"Parsing: {PBF_FILE}")
    handler = GeometryCollector()
    handler.apply_file(PBF_FILE, locations=True)

    for name, feats in handler.features.items():
        print(f"Saving {len(feats)} {name}...")
        save_geojson(name, feats)

    print("✅ Done. Check the 'extracted_features' folder.")


Parsing: Data_collection/nevada-latest.osm.pbf
Saving 3857 railways...
Saving 66431 water...
Saving 91429 footpaths...
Saving 3367 ramps...
Saving 3022 bike_lanes...
✅ Done. Check the 'extracted_features' folder.


In [2]:
import osmium
import geojson
import os

OUTPUT_DIR = "extracted_features"
PBF_FILE = "Data_collection/nevada-latest.osm.pbf"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

class BusStopExtractor(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.features = {
            'bus_stops': []
        }

    def node(self, n):
        if not n.location.valid():
            return
        tags = dict(n.tags)
        if tags.get('highway') == 'bus_stop' or tags.get('public_transport') in {'platform', 'stop_position'}:
            pt = geojson.Point((n.location.lon, n.location.lat))
            self.features['bus_stops'].append((pt, tags))

def save_geojson(name, features):
    fc = geojson.FeatureCollection([
        geojson.Feature(geometry=geom, properties=tags)
        for geom, tags in features
    ])
    with open(os.path.join(OUTPUT_DIR, f"{name}.geojson"), "w", encoding="utf-8") as f:
        geojson.dump(fc, f, indent=2)

if __name__ == "__main__":
    handler = BusStopExtractor()
    print(f"Parsing: {PBF_FILE}")
    handler.apply_file(PBF_FILE, locations=True)

    for name, feats in handler.features.items():
        print(f"Saving {len(feats)} {name}...")
        save_geojson(name, feats)


Parsing: Data_collection/nevada-latest.osm.pbf
Saving 1244 bus_stops...


In [4]:
import osmium
import geojson
import os

OUTPUT_DIR = "extracted_features"
PBF_FILE = "Data_collection/nevada-latest.osm.pbf"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

class TrafficLightExtractor(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.features = []

    def node(self, n):
        if not n.location.valid():
            return
        tags = dict(n.tags)
        if tags.get('highway') == 'traffic_signals':
            pt = geojson.Point((n.location.lon, n.location.lat))
            self.features.append(geojson.Feature(geometry=pt, properties=tags))

# Run extraction
handler = TrafficLightExtractor()
print(f"Parsing: {PBF_FILE}")
handler.apply_file(PBF_FILE, locations=True)

# Save GeoJSON
fc = geojson.FeatureCollection(handler.features)
with open(os.path.join(OUTPUT_DIR, "traffic_lights.geojson"), "w", encoding="utf-8") as f:
    geojson.dump(fc, f, indent=2)

print(f"✅ Saved {len(handler.features)} traffic lights.")


Parsing: Data_collection/nevada-latest.osm.pbf
✅ Saved 5493 traffic lights.


In [ ]:
import rasterio
import numpy as np
from shapely.geometry import LineString
import geopandas as gpd

# Load DEM
dem_path = "Data_collection/output.tin"
dem = rasterio.open(dem_path)

# Define a sample road line segment (lat/lon)
road_line = LineString([
    (-115.1405, 36.1695),  # Point A (before stop)
    (-115.1395, 36.1705)   # Point B (after stop)
])

# Reproject to DEM's CRS
gdf = gpd.GeoDataFrame(geometry=[road_line], crs="EPSG:4326").to_crs(dem.crs)
coords = list(gdf.geometry.iloc[0].coords)

# Sample elevation at points
elevations = []
for x, y in coords:
    row, col = dem.index(x, y)
    elev = dem.read(1)[row, col]
    elevations.append(elev)

# Calculate slope
delta_elev = elevations[-1] - elevations[0]
line_length_m = gdf.geometry.iloc[0].length
slope_percent = (delta_elev / line_length_m) * 100

print(f"📈 Slope along road segment: {slope_percent:.2f}%")
